# CS3807 – Deep Learning Laboratory
## Experiment 5 – Modified Part 2
### 5-Fold Cross-Validation, Final Evaluation and Additional Configurations

This notebook is fully self-contained. It intentionally avoids loading all images into a large NumPy array for cross-validation; fold membership is applied directly to the TFDS dataset to reduce RAM usage. All reported metrics are calculated at runtime.

In [ ]:
import os
import time
import random
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
NUM_CLASSES = 37
EPOCHS = 3
BATCH_SIZE = 24
OUTPUT_DIR = os.path.join(os.getcwd(), "Ex5_Modified_Outputs")
FIG_DIR = os.path.join(OUTPUT_DIR, "Figures")
os.makedirs(FIG_DIR, exist_ok=True)
print("Seed:", SEED)
print("TensorFlow:", tf.__version__)
print("Output directory:", OUTPUT_DIR)

In [ ]:
(train_raw, val_raw, test_raw), ds_info = tfds.load(
    "oxford_iiit_pet",
    split=["train[:80%]", "train[80%:]", "test"],
    as_supervised=True,
    with_info=True
)

print("Dataset loaded successfully.")
print("Number of classes:", ds_info.features["label"].num_classes)
print("Training samples:", tf.data.experimental.cardinality(train_raw).numpy())
print("Validation samples:", tf.data.experimental.cardinality(val_raw).numpy())
print("Test samples:", tf.data.experimental.cardinality(test_raw).numpy())
print("First five classes:", ds_info.features["label"].names[:5])

In [ ]:
def preprocess_image(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    return image, label

train_data = train_raw.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_data = val_raw.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
test_data = test_raw.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

train_data = train_data.shuffle(1024, seed=SEED, reshuffle_each_iteration=True).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_data = val_data.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_data = test_data.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for images, labels in train_data.take(1):
    print("Image batch shape:", images.shape)
    print("Label batch shape:", labels.shape)
    print("Image dtype:", images.dtype)

## 8. Memory-efficient 5-fold Cross-Validation

In [ ]:
# Create a fixed label array only. Images stay inside the tf.data pipeline.
y_labels=np.array([label.numpy() for _,label in train_raw])
indices=np.arange(len(y_labels))
skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
folds=list(skf.split(indices,y_labels))
print("Total training samples for CV:",len(indices))
print("Number of folds:",len(folds))
for i,(tr,va) in enumerate(folds,1): print(f"Fold {i}: train={len(tr)}, validation={len(va)}")

In [ ]:
def make_indexed_dataset(raw_ds, selected_indices, batch_size=24, training=False):
    selected_indices=np.asarray(selected_indices,dtype=np.int64)
    keys=tf.constant(selected_indices,dtype=tf.int64)
    values=tf.ones((len(selected_indices),),dtype=tf.int32)
    initializer=tf.lookup.KeyValueTensorInitializer(keys,values,key_dtype=tf.int64,value_dtype=tf.int32)
    table=tf.lookup.StaticHashTable(initializer,default_value=0)
    ds=raw_ds.enumerate()
    ds=ds.filter(lambda idx,data: tf.equal(table.lookup(idx),1))
    ds=ds.map(lambda idx,data: preprocess_image(data[0],data[1]),num_parallel_calls=tf.data.AUTOTUNE)
    if training: ds=ds.shuffle(min(len(selected_indices),1024),seed=SEED,reshuffle_each_iteration=True)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def build_cv_model(configuration):
    base=MobileNetV2(input_shape=(*IMG_SIZE,3),include_top=False,weights="imagenet")
    lr=5e-4
    if configuration=="Feature Extraction":
        base.trainable=False; dropout=0.0; use_bn=False
    elif configuration=="Dropout 0.30":
        base.trainable=False; dropout=0.30; use_bn=False
    elif configuration=="Batch Normalization":
        base.trainable=False; dropout=0.0; use_bn=True
    elif configuration=="Fine-Tuning":
        base.trainable=True; dropout=0.30; use_bn=False; lr=1e-5
        for layer in base.layers[:-30]: layer.trainable=False
        for layer in base.layers:
            if isinstance(layer,layers.BatchNormalization): layer.trainable=False
    else: raise ValueError(f"Unknown configuration: {configuration}")
    inputs=keras.Input(shape=(*IMG_SIZE,3)); x=layers.GlobalAveragePooling2D()(base(inputs,training=False))
    if use_bn: x=layers.BatchNormalization()(x)
    x=layers.Dense(160,activation="relu")(x)
    if dropout: x=layers.Dropout(dropout)(x)
    outputs=layers.Dense(NUM_CLASSES,activation="softmax")(x)
    model=keras.Model(inputs,outputs); model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),loss="sparse_categorical_crossentropy",metrics=["accuracy"]); return model

cv_configurations=["Feature Extraction","Dropout 0.30","Batch Normalization","Fine-Tuning"]
CV_EPOCHS=3
cv_results=[]
for config in cv_configurations:
    print("\n###",config,"###")
    fold_acc=[]; fold_time=[]
    for fold,(tr_idx,va_idx) in enumerate(folds,1):
        train_fold=make_indexed_dataset(train_raw,tr_idx,BATCH_SIZE,True)
        val_fold=make_indexed_dataset(train_raw,va_idx,BATCH_SIZE,False)
        model=build_cv_model(config); start=time.time(); h=model.fit(train_fold,validation_data=val_fold,epochs=CV_EPOCHS,verbose=1); elapsed=time.time()-start
        acc=max(h.history["val_accuracy"])*100; fold_acc.append(acc); fold_time.append(elapsed)
        print(f"Fold {fold}: {acc:.2f}% | {elapsed:.2f}s")
        del model,train_fold,val_fold,h; keras.backend.clear_session(); gc.collect()
    cv_results.append({"Configuration":config,"Fold 1":fold_acc[0],"Fold 2":fold_acc[1],"Fold 3":fold_acc[2],"Fold 4":fold_acc[3],"Fold 5":fold_acc[4],"Mean Accuracy (%)":float(np.mean(fold_acc)),"SD (%)":float(np.std(fold_acc)),"Mean Training Time (s)":float(np.mean(fold_time))})
cv_results_df=pd.DataFrame(cv_results)
cv_results_df

In [ ]:
best_cv_row=cv_results_df.loc[cv_results_df["Mean Accuracy (%)"].idxmax()]
best_cv_configuration=best_cv_row["Configuration"]
best_cv_mean=float(best_cv_row["Mean Accuracy (%)"])
best_cv_sd=float(best_cv_row["SD (%)"])
print("Selected configuration:",best_cv_configuration)
print(f"Mean CV accuracy: {best_cv_mean:.2f}% ± {best_cv_sd:.2f}%")

plt.figure(figsize=(10,6)); plt.bar(cv_results_df["Configuration"],cv_results_df["Mean Accuracy (%)"],yerr=cv_results_df["SD (%)"],capsize=5); plt.ylabel("Mean CV Accuracy (%)"); plt.title("Plot 13 – 5-Fold Cross-Validation Accuracy"); plt.xticks(rotation=20); plt.grid(axis="y",alpha=0.3); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_13_cv_accuracy.png"),dpi=300); plt.show()
cv_results_df.to_csv(os.path.join(OUTPUT_DIR,"cv_results.csv"),index=False)

## 9. Final Model Training and Independent Test Evaluation

In [ ]:
final_model=build_cv_model(best_cv_configuration)
start=time.time(); final_history=final_model.fit(train_data,validation_data=val_data,epochs=EPOCHS,verbose=1); final_training_time=time.time()-start
test_loss,test_accuracy=final_model.evaluate(test_data,verbose=1)

y_true=[]; y_pred=[]
for images,labels in test_data:
    pred=np.argmax(final_model.predict(images,verbose=0),axis=1)
    y_true.extend(labels.numpy()); y_pred.extend(pred)
y_true=np.asarray(y_true); y_pred=np.asarray(y_pred)
precision=precision_score(y_true,y_pred,average="macro",zero_division=0)
recall=recall_score(y_true,y_pred,average="macro",zero_division=0)
f1=f1_score(y_true,y_pred,average="macro",zero_division=0)
print(f"Test accuracy: {test_accuracy*100:.2f}%")
print(f"Macro precision: {precision*100:.2f}%")
print(f"Macro recall: {recall*100:.2f}%")
print(f"Macro F1-score: {f1*100:.2f}%")
print(f"Training time: {final_training_time:.2f} s")
print(f"Parameters: {final_model.count_params():,}")

In [ ]:
final_results=pd.DataFrame({"Metric":["Mean CV Accuracy","CV Standard Deviation","Test Accuracy","Macro Precision","Macro Recall","Macro F1-score","Training Time (s)","Number of Parameters"],"Value":[f"{best_cv_mean:.2f}%",f"{best_cv_sd:.2f}%",f"{test_accuracy*100:.2f}%",f"{precision*100:.2f}%",f"{recall*100:.2f}%",f"{f1*100:.2f}%",f"{final_training_time:.2f}",f"{final_model.count_params():,}"]})
final_results

In [ ]:
plt.figure(figsize=(10,6)); plt.plot(np.array(final_history.history["accuracy"])*100,marker="o",label="Training Accuracy"); plt.plot(np.array(final_history.history["val_accuracy"])*100,marker="o",label="Validation Accuracy"); plt.xlabel("Epoch"); plt.ylabel("Accuracy (%)"); plt.title("Final Model Training and Validation Accuracy"); plt.legend(); plt.grid(True); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"final_model_accuracy.png"),dpi=300); plt.show()

class_names=ds_info.features["label"].names
cm=confusion_matrix(y_true,y_pred)
plt.figure(figsize=(16,14)); sns.heatmap(cm,annot=False,cmap="Blues",xticklabels=class_names,yticklabels=class_names); plt.xlabel("Predicted Class"); plt.ylabel("True Class"); plt.title("Plot 14 – Confusion Matrix: Final Model"); plt.xticks(rotation=90,fontsize=7); plt.yticks(rotation=0,fontsize=7); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"plot_14_confusion_matrix.png"),dpi=300); plt.show()
print(classification_report(y_true,y_pred,target_names=class_names,zero_division=0))
final_results.to_csv(os.path.join(OUTPUT_DIR,"final_model_results.csv"),index=False)

## 10. Additional Exercise
Two additional configurations are evaluated with the same five folds. Unlike the reference implementation, the fold datasets remain memory-efficient and the results are calculated dynamically.

In [ ]:
additional_configurations={
    "Additional C1":{"learning_rate":2e-4,"dropout_rate":0.40,"batch_size":16,"fine_tuning":False},
    "Additional C2":{"learning_rate":5e-5,"dropout_rate":0.20,"batch_size":48,"fine_tuning":True},
}

def build_additional_model(cfg):
    base=MobileNetV2(input_shape=(*IMG_SIZE,3),include_top=False,weights="imagenet")
    if cfg["fine_tuning"]:
        base.trainable=True
        for layer in base.layers[:-15]: layer.trainable=False
        for layer in base.layers:
            if isinstance(layer,layers.BatchNormalization): layer.trainable=False
    else: base.trainable=False
    inputs=keras.Input(shape=(*IMG_SIZE,3)); x=layers.GlobalAveragePooling2D()(base(inputs,training=False)); x=layers.Dense(160,activation="relu")(x); x=layers.Dropout(cfg["dropout_rate"])(x); outputs=layers.Dense(NUM_CLASSES,activation="softmax")(x)
    model=keras.Model(inputs,outputs); model.compile(optimizer=keras.optimizers.Adam(learning_rate=cfg["learning_rate"]),loss="sparse_categorical_crossentropy",metrics=["accuracy"]); return model

additional_results=[]
for name,cfg in additional_configurations.items():
    accs=[]; times=[]
    for fold,(tr_idx,va_idx) in enumerate(folds,1):
        tr=make_indexed_dataset(train_raw,tr_idx,cfg["batch_size"],True); va=make_indexed_dataset(train_raw,va_idx,cfg["batch_size"],False)
        model=build_additional_model(cfg); start=time.time(); h=model.fit(tr,validation_data=va,epochs=CV_EPOCHS,verbose=0); elapsed=time.time()-start
        accs.append(max(h.history["val_accuracy"])*100); times.append(elapsed); print(f"{name} fold {fold}: {accs[-1]:.2f}%")
        del model,tr,va,h; keras.backend.clear_session(); gc.collect()
    additional_results.append({"Configuration":name,"Learning Rate":cfg["learning_rate"],"Dropout Rate":cfg["dropout_rate"],"Batch Size":cfg["batch_size"],"Fine-Tuning":cfg["fine_tuning"],"Fold 1":accs[0],"Fold 2":accs[1],"Fold 3":accs[2],"Fold 4":accs[3],"Fold 5":accs[4],"Mean Accuracy (%)":np.mean(accs),"SD (%)":np.std(accs),"Mean Training Time (s)":np.mean(times)})
additional_cv_results_df=pd.DataFrame(additional_results)
additional_cv_results_df

In [ ]:
comparison_results=pd.concat([cv_results_df[cv_results_df["Configuration"]==best_cv_configuration][["Configuration","Mean Accuracy (%)","SD (%)","Mean Training Time (s)"]],additional_cv_results_df[["Configuration","Mean Accuracy (%)","SD (%)","Mean Training Time (s)"]]],ignore_index=True)
comparison_results.to_csv(os.path.join(OUTPUT_DIR,"selected_vs_additional.csv"),index=False)
comparison_results

plt.figure(figsize=(10,6)); plt.bar(comparison_results["Configuration"],comparison_results["Mean Accuracy (%)"],yerr=comparison_results["SD (%)"],capsize=5); plt.ylabel("Mean CV Accuracy (%)"); plt.title("Additional Exercise – Selected vs Additional Configurations"); plt.xticks(rotation=20); plt.grid(axis="y",alpha=0.3); plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR,"additional_exercise_comparison.png"),dpi=300); plt.show()

## Experiment 5 runtime outcome
The notebook intentionally does not contain fixed accuracy values. The selected configuration, cross-validation statistics and final test metrics are created from the actual run and saved as CSV files in `Ex5_Modified_Outputs`.

In [ ]:
print("===== FINAL RUNTIME SUMMARY =====")
print("Selected CV configuration:",best_cv_configuration)
print(f"Mean CV accuracy: {best_cv_mean:.2f}% ± {best_cv_sd:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}% | Recall: {recall*100:.2f}% | F1: {f1*100:.2f}%")
print("Saved files:",sorted(os.listdir(OUTPUT_DIR))[:10])